In [1]:
import os
import nibabel as nib
import numpy as np 
from collections import OrderedDict
import json
from pathlib import Path
from tabulate import tabulate

from utils.helper import convert_nrrd_to_nifti, create_folder, plot_all_slices, plot_histogram, generate_binary_image, adjust_affine_for_spacing_and_origin, save_binary_image_with_adjusted_origin, make_if_dont_exist
from utils.metrics import dice_score_per_class, hausdorff_distance_per_class, ravd_per_class

In [2]:

# define dataset path
BASE_PATH = Path('./').resolve()
DATA_PATH = BASE_PATH / 'dataset'

project_name = 'HCFC1' #change here for different task name
task_name = 'Dataset002_' + project_name 

TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTr'
GT_TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTr'
TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTs'
GT_TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTs'
PREDICTION_RESULTS_PATH  = BASE_PATH / 'dataset/nnUNet_Prediction_Results' / task_name
TASK_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name 

# setup environment variables
nnUNet_raw = BASE_PATH / 'dataset/nnUNet_raw_data'
nnUNet_preprocessed = BASE_PATH / 'dataset/nnUNet_preprocessed'
nnUNet_results = BASE_PATH / 'dataset/nnUNet_results'

In [3]:
def read_nifti(path):
    img = nib.load(path)

    return img.get_fdata(),img.shape

In [4]:
ds_score = []
hd_score = []
havd_score = []

# Specify the path to your JSON file
json_file_path = TASK_PATH / 'dataset.json'

# Read the JSON file
with open(json_file_path, 'r') as file:
    data = json.load(file)

labels = data['labels']
head = ['Fold'] + ['Volume ID'] + list(labels.keys())

ds_score.append(head)
hd_score.append(head)
havd_score.append(head)

gtPath = GT_TRAINING_DATASET_PATH
for i in range(5):
    
    imagePath = f"/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/fold_{i}/validation"

    all_files = os.listdir(imagePath)
    for file in all_files:
        if file.endswith(".nii.gz"):
            imgP = os.path.join(imagePath, file)
            gtP = os.path.join(gtPath, file)
            print(imgP)
            print(gtP)
            imgData, i_ = read_nifti(imgP)
            gtData, g_ = read_nifti(gtP)
            newname = file.split("_")[0]
            print(newname)
            print(i_)
            print(g_)

            concatenated_array = np.concatenate(([i],[newname], dice_score_per_class(imgData,gtData,25)))
            concatenated_array = np.transpose(concatenated_array)


            ds_score.append(concatenated_array)
        # hd_score.append(hausdorff_distance_per_class(imgData,gtData,25))
        # havd_score.append(ravd_per_class(imgData,gtData,25))


/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/fold_0/validation/NG4111_RCL5.nii.gz
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr/NG4111_RCL5.nii.gz
NG4111
(239, 879, 455)
(239, 879, 455)
/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/fold_0/validation/NG4108_RCL5.nii.gz
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr/NG4108_RCL5.nii.gz
NG4108
(453, 789, 402)
(453, 789, 402)
/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/fold_0/validation/NG4116_RCL5.nii.gz
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr/NG4116_RCL5.nii.gz
NG4116
(366, 939, 405)
(366, 939, 405)
/user1/ngm

In [5]:
print(tabulate(ds_score, tablefmt="grid"))

+------+-----------+------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+
| Fold | Volume ID | background | CTX+   | cc+    | CPu    | DG     | HP     | RHP    | A      | ig     | fi     | f      | st     | ic     | och    | ac     | fr     | Hb     | TH     | HY     | MB     | P      | MY     | TCB    | V      | OB     |
+------+-----------+------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+
| 0    | NG4111    | 0.9944     | 0.9603 | 0.8676 | 0.9480 | 0.9300 | 0.8999 | 0.8832 | 0.6847 | 0.5727 | 0.9103 | 0.7187 | 0.7564 | 0.8244 | 0.8894 | 0.8329 | 0.7702 | 0.8752 | 0.9392 | 0.8773 | 0.9533 | 0.9427 | 0.9721 | 0.9848 | 0.8937 | 0.9616 |


In [6]:
def transpose_table(table):
    transposed_table = []

    # Get the header row and remove it from the table
    header = table.pop(0)
    # Initialize transposed table with the Volume ID column
    transposed_table.append(["Fold"] + [row[0] for row in table])

    # Transpose the table
    for i in range(1, len(header)):
        transposed_row = [header[i]]
        for j in range(len(table)):
            transposed_row.append(table[j][i])
        transposed_table.append(transposed_row)

    return transposed_table


# # Transpose the table
# transposed_table = transpose_table(ds_score)

# print(tabulate(transposed_table, tablefmt="grid"))


In [7]:
import pandas as pd

# Define your array
data = transpose_table(ds_score) 

# Define column names
columns = ["Volume ID"] + [f"Value_{i}" for i in range(1, len(data[0]))]

# Create DataFrame
df = pd.DataFrame(data)

# Save DataFrame to Excel
df.to_excel(BASE_PATH/"data/validation_dice_v1.xlsx", index=False)

print("DataFrame saved to output.xlsx")


DataFrame saved to output.xlsx


In [9]:
imgData, i_ = read_nifti("/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/imagesTr/NG4112_RCL5_0000.nii.gz")
print("Image: ", i_)
gtData,g_ = read_nifti("/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr/NG4112_RCL5.nii.gz")
print("GT: ",g_)

Image:  (280, 794, 416)
GT:  (280, 794, 416)
